# Module 4: Search in Azure DocumentDB

**Time**: ~75 min  
**Environment**: Jupyter notebook in VS Code

This notebook is fully runnable. Enter your Azure DocumentDB connection string and OpenAI API key in Step 0, then run each cell in order. The notebook creates embeddings for the sample documents, stores them in Azure DocumentDB, creates vector and full-text indexes, and runs vector, BM25, fuzzy, phrase, and hybrid search.

> Full-text search in Azure DocumentDB is currently in gated preview. DiskANN vector search requires an M30 or higher cluster tier.


## Step 0: Connect and configure embeddings

This cell installs dependencies if needed, accepts the DocumentDB connection string and OpenAI API key, and creates both clients. The default embedding model is `text-embedding-3-small`.

In [ ]:
import importlib.util, subprocess, sys, os, getpass
for package in ["pymongo", "openai"]:
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
from pymongo import MongoClient
from openai import OpenAI

connection_string = os.environ.get("DOCUMENTDB_CONNECTION_STRING") or getpass.getpass("Paste Azure DocumentDB connection string: ")
openai_api_key = os.environ.get("OPENAI_API_KEY") or getpass.getpass("Paste OpenAI API key: ")
embedding_model = os.environ.get("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")

client = MongoClient(connection_string)
db = client["docdbworkshop"]
collection = db["workshop_content"]
openai_client = OpenAI(api_key=openai_api_key)
print(db.command({"ping": 1}))
print("Embedding model:", embedding_model)

## Step 1: Generate embeddings and load sample documents

This cell calls the OpenAI embeddings API for each sample document body, stores the resulting vector in the `embedding` field, and inserts the documents into Azure DocumentDB.

In [ ]:
def create_embedding(text: str) -> list[float]:
    response = openai_client.embeddings.create(model=embedding_model, input=text)
    return response.data[0].embedding

source_docs = [
    {"_id":"doc-search-001","title":"DiskANN vector indexing","category":"vector","body":"Azure DocumentDB supports DiskANN vector indexes for high recall semantic similarity search over embeddings stored with documents.","sku":"SEARCH-VEC-001"},
    {"_id":"doc-search-002","title":"BM25 keyword search","category":"full-text","body":"Azure DocumentDB full-text search ranks keyword matches with BM25 and exposes scores through searchScore metadata.","sku":"SEARCH-FTS-001"},
    {"_id":"doc-search-003","title":"Hybrid search with RRF","category":"hybrid","body":"Hybrid search combines BM25 keyword results with vector results and fuses the ranked lists using Reciprocal Rank Fusion.","sku":"SEARCH-HYB-001"},
    {"_id":"doc-search-004","title":"RAG grounding","category":"rag","body":"Retrieval augmented generation retrieves relevant chunks from Azure DocumentDB and grounds the model answer in that context.","sku":"RAG-PIPE-001"},
    {"_id":"doc-search-005","title":"Operational filtering","category":"filters","body":"Search applications often filter by status, tenant, region, stock, or category after the search stage narrows candidate documents.","sku":"SEARCH-FLT-001"}
]

collection.drop()
for doc in source_docs:
    doc["embedding"] = create_embedding(doc["body"])
collection.insert_many(source_docs)
embedding_dimensions = len(source_docs[0]["embedding"])
print("Loaded documents:", collection.count_documents({}))
print("Embedding dimensions:", embedding_dimensions)

## Step 2: Create vector and full-text indexes

The vector index uses `cosmosSearch` and the actual embedding dimension returned by OpenAI. The full-text index uses `createSearchIndexes` over the `body` field.

**STUDENT EXERCISE:** you will complete the index command in the next cell. Compare with the matching `after` notebook if you get stuck.


In [ ]:
# STUDENT EXERCISE: create both indexes.
# 1. Vector: createIndexes on workshop_content with key {"embedding": "cosmosSearch"}.
# 2. Full-text: createSearchIndexes named idx_body_fts over body.
# Use embedding_dimensions for the vector dimensions.

vector_index_result = {"ok": 0, "message": "TODO: create idx_embedding_diskann"}
full_text_index_result = {"ok": 0, "message": "TODO: create idx_body_fts"}
{"vector": vector_index_result, "fullText": full_text_index_result}


## Step 3: Generate a query embedding and run vector search

The query text is embedded with the same model, then used as the `vector` in `$search.cosmosSearch`.

**STUDENT EXERCISE:** complete the vector retrieval query. Expected result: top chunks/documents should relate to RAG, vector, or hybrid search.


In [ ]:
search_text = "semantic retrieval for RAG"
query_vector = create_embedding(search_text)

# STUDENT EXERCISE: replace this with a $search.cosmosSearch pipeline.
# Include path="embedding", query=query_vector, k=3, and lSearch=40.
vector_pipeline = [{"$limit": 0}]

list(collection.aggregate(vector_pipeline))


## Step 4: Run BM25 full-text search

This query uses `$search.text` against the named full-text index and projects BM25 relevance scores with `$meta: "searchScore"`.

**STUDENT EXERCISE:** complete the search or hybrid retrieval snippet in the next cell. Notice where `$limit` belongs and how `searchScore` is projected.


In [ ]:
bm25_results = list(collection.aggregate([
    {"$search": {"index": "idx_body_fts", "text": {"query": "BM25 ranking", "path": "body"}}},
    {"$limit": 5},
    {"$project": {"_id": 0, "title": 1, "body": 1, "score": {"$meta": "searchScore"}}}
]))
bm25_results

## Step 5: Run fuzzy and phrase search

Fuzzy search tolerates typos with `maxEdits`. Phrase search requires ordered terms, with `slop` controlling how close the words must be.

In [ ]:
fuzzy_results = list(collection.aggregate([
    {"$search": {"index": "idx_body_fts", "text": {"query": "retrival augmentd genration", "path": "body", "fuzzy": {"maxEdits": 1}}}},
    {"$limit": 5},
    {"$project": {"_id": 0, "title": 1, "score": {"$meta": "searchScore"}}}
]))
phrase_results = list(collection.aggregate([
    {"$search": {"index": "idx_body_fts", "phrase": {"query": "Reciprocal Rank Fusion", "path": "body", "slop": 0}}},
    {"$limit": 5},
    {"$project": {"_id": 0, "title": 1, "score": {"$meta": "searchScore"}}}
]))
{"fuzzy": fuzzy_results, "phrase": phrase_results}

## Step 6: Run hybrid search

Hybrid search embeds the user query, runs vector and BM25 retrieval, and fuses both ranked lists with RRF.

In [ ]:
search_text = "semantic retrieval for RAG"
query_vector = create_embedding(search_text)

# STUDENT EXERCISE: replace this with a $search.cosmosSearch pipeline.
# Include path="embedding", query=query_vector, k=3, and lSearch=40.
vector_pipeline = [{"$limit": 0}]

list(collection.aggregate(vector_pipeline))
